In [ ]:
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklite import LazyExport

warnings.filterwarnings("ignore")

DATA_DIR = "."
CLASS_FILES = {
    "Baseline": "aquafina_baseline_aug50.csv",
    "CuCl2": "aquafina_cucl2_aug50.csv",
    "NiCl2": "aquafina_nicl2_aug50.csv",
    "KNO3": "aquafina_kno3_aug50.csv",
    "NaNO3": "aquafina_nano3_aug50.csv",
    "Pb(NO3)2": "aquafina_pbno3_aug50.csv",
    "FeCl3": "aquafina_fecl3_aug50.csv",
    "CuCl2+NiCl2": "aquafina_cucl2_nicl2_aug50.csv",
    "KNO3+NaNO3": "aquafina_kno3_nano3_aug50.csv",
    "Pb(NO3)2+FeCl3": "aquafina_pbno3_fecl3_aug50.csv",
}
CLASSES = list(CLASS_FILES.keys())
PLOT_LABELS = ["Base", "CuCl2", "NiCl2", "KNO3", "NaNO3", "Pb", "FeCl3", "Cu+Ni", "K+Na", "Pb+Fe"]

# load one CSV per class
data_by_class = {}
feature_columns = None
label_columns = {"label", "Label", "class"}

for class_name, filename in CLASS_FILES.items():
    df = pd.read_csv(f"{DATA_DIR}/{filename}")
    if feature_columns is None:
        feature_columns = [col for col in df.columns if col not in label_columns]
    data_by_class[class_name] = df

# build feature matrix and matching labels
feature_rows = []
labels = []
for class_name in CLASSES:
    class_features = data_by_class[class_name][feature_columns].to_numpy()
    feature_rows.append(class_features)
    labels.extend([class_name] * len(class_features))

X_real = np.vstack(feature_rows)
y_real = np.array(labels)
print(f"{len(X_real)} samples, {len(feature_columns)} features")


In [ ]:
def augment_fold(X_train, y_train, target_per_class=50, seed=None):
    rng = np.random.default_rng(seed)
    extra_X = []
    extra_y = []

    for class_name in np.unique(y_train):
        class_mask = y_train == class_name
        class_samples = X_train[class_mask]
        n_real = len(class_samples)
        n_to_add = max(0, target_per_class - n_real)

        extra_X.append(class_samples)
        extra_y.extend([class_name] * n_real)

        if n_to_add > 0:
            feature_std = class_samples.std(axis=0)
            feature_std[feature_std == 0] = 1e-6

            random_rows = rng.integers(0, n_real, size=n_to_add)
            noise = rng.standard_normal((n_to_add, class_samples.shape[1])) * feature_std
            synthetic = class_samples[random_rows] + noise

            extra_X.append(synthetic)
            extra_y.extend([class_name] * n_to_add)

    return np.vstack(extra_X), np.array(extra_y)


model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = np.empty(len(y_real), dtype=object)
probabilities = np.zeros((len(y_real), len(CLASSES)))

for fold_number, (train_idx, test_idx) in enumerate(cv.split(X_real, y_real)):
    X_train, y_train = augment_fold(X_real[train_idx], y_real[train_idx], seed=42 + fold_number)
    X_test = X_real[test_idx]
    y_test = y_real[test_idx]

    model.fit(X_train, y_train)
    predictions[test_idx] = model.predict(X_test)

    fold_probs = model.predict_proba(X_test)
    for col, class_name in enumerate(model.classes_):
        class_index = CLASSES.index(class_name)
        probabilities[test_idx, class_index] = fold_probs[:, col]

    fold_acc = accuracy_score(y_test, predictions[test_idx])
    print(f"fold {fold_number + 1}: acc={fold_acc:.3f}")

accuracy = accuracy_score(y_real, predictions)
f1_macro = f1_score(y_real, predictions, average="macro")
labels_one_hot = label_binarize(y_real, classes=CLASSES)
auc_macro = roc_auc_score(labels_one_hot, probabilities, average="macro", multi_class="ovr")
print(f"acc={accuracy:.3f}  f1={f1_macro:.3f}  auc={auc_macro:.3f}")


In [ ]:
precision = precision_score(y_real, predictions, average=None, labels=CLASSES)
recall = recall_score(y_real, predictions, average=None, labels=CLASSES)
f1_scores = f1_score(y_real, predictions, average=None, labels=CLASSES)

auc_scores = []
for i in range(len(CLASSES)):
    score = roc_auc_score(labels_one_hot[:, i], probabilities[:, i])
    auc_scores.append(score)
auc_scores = np.array(auc_scores)

metrics = pd.DataFrame({
    "class": CLASSES,
    "prec": np.round(precision, 3),
    "rec": np.round(recall, 3),
    "f1": np.round(f1_scores, 3),
    "auc": np.round(auc_scores, 3),
})
print(metrics.to_string(index=False))


In [ ]:
cm = confusion_matrix(y_real, predictions, labels=CLASSES)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(PLOT_LABELS)))
ax.set_yticks(range(len(PLOT_LABELS)))
ax.set_xticklabels(PLOT_LABELS, rotation=45, ha="right")
ax.set_yticklabels(PLOT_LABELS)
ax.set_title(f"confusion matrix ({accuracy * 100:.1f}%)")

for row in range(len(CLASSES)):
    for col in range(len(CLASSES)):
        count = cm[row, col]
        if count:
            ax.text(col, row, count, ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

sorted_by_f1 = np.argsort(f1_scores)[::-1]
sorted_labels = [PLOT_LABELS[i] for i in sorted_by_f1]
sorted_f1 = f1_scores[sorted_by_f1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(sorted_labels, sorted_f1)
ax.set_xlim(0, 1)
ax.set_title("per-class f1")
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for i, label in enumerate(PLOT_LABELS):
    false_positive_rate, true_positive_rate, _ = roc_curve(labels_one_hot[:, i], probabilities[:, i])
    ax.plot(false_positive_rate, true_positive_rate, label=f"{label} ({auc_scores[i]:.2f})")

ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title(f"roc (macro auc={auc_macro:.3f})")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.show()


In [ ]:
X_full, y_full = augment_fold(X_real, y_real, seed=42)

numeric_labels = []
for class_name in y_full:
    numeric_labels.append(CLASSES.index(class_name))
numeric_labels = np.array(numeric_labels)

model.fit(X_full, numeric_labels)

joblib.dump(model, "safesip_rf.joblib")

with open("safesip_feat_cols.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

with open("safesip_classes.json", "w") as f:
    json.dump(CLASSES, f, indent=2)

LazyExport(model).save("safesip_rf.json")
print("wrote safesip_rf.joblib, safesip_feat_cols.json, safesip_classes.json, safesip_rf.json")
